In [1]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [2]:
!pip install torch torchvision scikit-learn tqdm

In [4]:
!ls

sample_data  SmartRoadAI_Dataset.zip


In [6]:
!unzip -q SmartRoadAI_Dataset.zip

In [8]:
!ls SmartRoadAI_Dataset


normal	potholes


In [9]:
import os

normal_count = len(os.listdir("SmartRoadAI_Dataset/normal"))
pothole_count = len(os.listdir("SmartRoadAI_Dataset/potholes"))

print("Normal Images :", normal_count)
print("Pothole Images:", pothole_count)
print("Total Images  :", normal_count + pothole_count)

Normal Images : 352
Pothole Images: 329
Total Images  : 681


In [10]:
import os
from sklearn.model_selection import train_test_split

normal_images = [
    os.path.join("SmartRoadAI_Dataset/normal", img)
    for img in os.listdir("SmartRoadAI_Dataset/normal")
]

pothole_images = [
    os.path.join("SmartRoadAI_Dataset/potholes", img)
    for img in os.listdir("SmartRoadAI_Dataset/potholes")
]

# Normal split
normal_train, normal_temp = train_test_split(
    normal_images,
    test_size=0.10,
    random_state=42
)

normal_val, normal_test = train_test_split(
    normal_temp,
    test_size=0.50,
    random_state=42
)

# Pothole split
pothole_train, pothole_temp = train_test_split(
    pothole_images,
    test_size=0.10,
    random_state=42
)

pothole_val, pothole_test = train_test_split(
    pothole_temp,
    test_size=0.50,
    random_state=42
)

print("TRAIN")
print("Normal  :", len(normal_train))
print("Pothole :", len(pothole_train))

print("\nVALIDATION")
print("Normal  :", len(normal_val))
print("Pothole :", len(pothole_val))

print("\nTEST")
print("Normal  :", len(normal_test))
print("Pothole :", len(pothole_test))

TRAIN
Normal  : 316
Pothole : 296

VALIDATION
Normal  : 18
Pothole : 16

TEST
Normal  : 18
Pothole : 17


In [11]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# Better augmentation for training
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor()
])

# Validation/Test transforms
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])


class PotholeDataset(Dataset):

    def __init__(self, image_paths, label, transform):
        self.image_paths = image_paths
        self.label = label
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        image = Image.open(
            self.image_paths[idx]
        ).convert("RGB")

        image = self.transform(image)

        return image, self.label


# Create datasets
train_dataset = torch.utils.data.ConcatDataset([
    PotholeDataset(normal_train, 0, train_transform),
    PotholeDataset(pothole_train, 1, train_transform)
])

val_dataset = torch.utils.data.ConcatDataset([
    PotholeDataset(normal_val, 0, test_transform),
    PotholeDataset(pothole_val, 1, test_transform)
])

test_dataset = torch.utils.data.ConcatDataset([
    PotholeDataset(normal_test, 0, test_transform),
    PotholeDataset(pothole_test, 1, test_transform)
])

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print("Train Images:", len(train_dataset))
print("Validation Images:", len(val_dataset))
print("Test Images:", len(test_dataset))

Train Images: 612
Validation Images: 34
Test Images: 35


In [12]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

model = models.resnet18(
    weights="DEFAULT"
)

num_features = model.fc.in_features

model.fc = nn.Linear(
    num_features,
    2
)

model = model.to(device)

print("Model Loaded Successfully")

Device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 118MB/s]


Model Loaded Successfully


In [13]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001
)

print("Loss Function and Optimizer Ready")

Loss Function and Optimizer Ready


In [14]:
EPOCHS = 25

best_val_accuracy = 0.0

print("Training Config Ready")

Training Config Ready


In [15]:
for epoch in range(EPOCHS):

    # TRAINING
    model.train()

    train_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    # VALIDATION
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(
                outputs,
                1
            )

            total += labels.size(0)

            correct += (
                predicted == labels
            ).sum().item()

    val_accuracy = (
        100 * correct / total
    )

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss: {train_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.2f}%"
    )

    # Save best model
    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy

        torch.save(
            model.state_dict(),
            "pothole_model_v2.pth"
        )

        print(
            f"Best Model Saved "
            f"({val_accuracy:.2f}%)"
        )

Epoch 1/25 | Loss: 3.3025 | Val Accuracy: 97.06%
Best Model Saved (97.06%)
Epoch 2/25 | Loss: 0.6629 | Val Accuracy: 97.06%
Epoch 3/25 | Loss: 1.8670 | Val Accuracy: 97.06%
Epoch 4/25 | Loss: 0.9991 | Val Accuracy: 100.00%
Best Model Saved (100.00%)
Epoch 5/25 | Loss: 0.3409 | Val Accuracy: 100.00%
Epoch 6/25 | Loss: 0.1865 | Val Accuracy: 100.00%
Epoch 7/25 | Loss: 0.1705 | Val Accuracy: 100.00%
Epoch 8/25 | Loss: 0.3308 | Val Accuracy: 100.00%
Epoch 9/25 | Loss: 2.6372 | Val Accuracy: 97.06%
Epoch 10/25 | Loss: 0.6661 | Val Accuracy: 100.00%
Epoch 11/25 | Loss: 0.7681 | Val Accuracy: 97.06%
Epoch 12/25 | Loss: 0.3402 | Val Accuracy: 97.06%
Epoch 13/25 | Loss: 0.1831 | Val Accuracy: 100.00%
Epoch 14/25 | Loss: 0.0637 | Val Accuracy: 97.06%
Epoch 15/25 | Loss: 2.9026 | Val Accuracy: 97.06%
Epoch 16/25 | Loss: 0.9566 | Val Accuracy: 97.06%
Epoch 17/25 | Loss: 0.1448 | Val Accuracy: 100.00%
Epoch 18/25 | Loss: 0.1850 | Val Accuracy: 94.12%
Epoch 19/25 | Loss: 0.1518 | Val Accuracy: 94.12

In [16]:
model.load_state_dict(
    torch.load("pothole_model_v2.pth")
)

model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(
            outputs,
            1
        )

        total += labels.size(0)

        correct += (
            predicted == labels
        ).sum().item()

test_accuracy = (
    100 * correct / total
)

print(
    f"Test Accuracy: {test_accuracy:.2f}%"
)

Test Accuracy: 100.00%


In [17]:
from google.colab import files

files.download("pothole_model_v2.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>